# Role 03 model integration: run AFTER 04
Numbers identify team roles, not execution order. Run 01 cleaning → 02 graph (when integrated) and 04 behaviour → 03 ML. First generate both behaviour files with notebooks/04_behaviour_features.ipynb or python -m src.generate_behaviour. This notebook validates their hashes and refuses missing/stale outputs. The stratified 80/20 split, seed 0, remains unchanged. Person 1's calendar features are included from her cleaned table; 04 does not calculate them.


## Behaviour integration and validation
Keep the supplied stratified 80/20 labelled-train split (random_state=0). Features use label-free history within the full train split before row holdout, matching the starter baseline. This is an offline/transductive holdout, not a chronological forecast or wallet-isolated evaluation. Train and test histories remain separate; the unlabelled organiser validation folder is optional for a smoke check and is not used for AP. Six behaviour fields use strictly earlier timestamps. Missing feature values stay missing in exported tables; a -1 sentinel plus missing indicators is applied only inside the model pipeline, fitted on the training fold. Generated outputs go to ignored outputs/. Run this notebook from src/.


# Decode & Detect: starter notebook

This notebook trains a simple detector you can build on:

1. Load the payments.
2. Turn each payment into a few **features**: simple rules such as "the receiver sends the money on within an hour".
3. Train a **logistic regression** on the labelled `train/` data. It learns how much each rule matters.
4. Score the test data and write `predictions.csv`, ready to upload at https://hackathon.ukcbt.org.

To improve your score, add your own rules in step 2 and run the notebook again.

**Setup:** put this notebook next to the `train/` folder and `payments.py` from the training download, and unzip the test download into a folder called `test-data/`. Then install the libraries:

```sh
pip install pandas pyarrow scikit-learn jupyter
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from behaviour_payments import connect_cleaned_payments
import sys

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(OUTPUT_DIR.resolve().parent))
from src.generate_behaviour import load_for_ml

TRAIN_DIR = Path("../data/raw/train")
TEST_DIR = Path("../data/raw/test")  # where you unzipped the test download

## 1. Load the payments

`load_payments` (from the download) gives one row per transaction: who sent it (`FROM_ADDRESS`), who received it (`TO_ADDRESS`), when (`TIMESTAMP`, in seconds) and its value in US dollars (`VALUE`), using the hourly reference prices.

In [ ]:
def load(directory):
    name = "train_cleaned.parquet" if directory == TRAIN_DIR else "test_cleaned.parquet"
    cleaned = pd.read_parquet(OUTPUT_DIR.resolve().parent / "data/processed" / name)
    connected = connect_cleaned_payments(cleaned, directory)
    # Preserve the original holdout assignment despite sorted cleaning exports.
    raw_order = pd.read_parquet(directory / "transactions.parquet", columns=["TRANSACTION_HASH"]).TRANSACTION_HASH
    return connected.set_index("TRANSACTION_HASH").loc[raw_order].reset_index()


train = load(TRAIN_DIR)
labels = pd.read_csv(TRAIN_DIR / "labels.csv", usecols=["TRANSACTION_HASH", "is_suspicious"])
train = train.merge(labels, on="TRANSACTION_HASH", validate="one_to_one")
print(f"{len(train):,} training payments, {train.is_suspicious.sum():,} suspicious")
train.head()

## 2. Features: simple rules

Each feature is one simple idea about suspicious money movement. The model only sees these numbers, so **this is where your ideas go**.

In [ ]:
HOUR, TEN_MINUTES = 3600, 600


def count_in_window(times_by_account, account, t, before, after):
    """How many of an account's payments fall between t - before and t + after."""
    times = times_by_account.get(account)
    if times is None:
        return 0
    return int(np.searchsorted(times, t + after, "right") - np.searchsorted(times, t - before, "left"))


def make_features(df):
    df = df.sort_values("TIMESTAMP").reset_index(drop=True)
    t = df.TIMESTAMP.astype(int).to_numpy()
    value = df.VALUE.to_numpy()
    sends = {a: g.TIMESTAMP.astype(int).to_numpy() for a, g in df.groupby("FROM_ADDRESS")}
    gets = {a: g.TIMESTAMP.astype(int).to_numpy() for a, g in df.groupby("TO_ADDRESS")}

    f = pd.DataFrame({"TRANSACTION_HASH": df.TRANSACTION_HASH})

    # Rule: bigger payments are more interesting (log keeps huge ones in range).
    f["log_value"] = np.log1p(value)

    # Rule: how busy are the sender and the receiver over the whole week?
    f["sender_payments"] = df.FROM_ADDRESS.map(df.FROM_ADDRESS.value_counts())
    f["receiver_payments"] = df.TO_ADDRESS.map(df.TO_ADDRESS.value_counts())

    # Rule: the receiver sends money on within an hour (a "relay" or "mule").
    f["receiver_forwards_1h"] = [
        count_in_window(sends, a, ti, 0, HOUR) > 0 for a, ti in zip(df.TO_ADDRESS, t)
    ]

    # Rule: the sender only just received money (it is passing it along).
    f["sender_received_1h_before"] = [
        count_in_window(gets, a, ti, HOUR, 0) > 0 for a, ti in zip(df.FROM_ADDRESS, t)
    ]

    # Rule: fan-out / fan-in, many payments from one sender or into one
    # receiver within ten minutes.
    f["sender_burst_10m"] = [
        count_in_window(sends, a, ti, TEN_MINUTES, TEN_MINUTES) for a, ti in zip(df.FROM_ADDRESS, t)
    ]
    f["receiver_burst_10m"] = [
        count_in_window(gets, a, ti, TEN_MINUTES, TEN_MINUTES) for a, ti in zip(df.TO_ADDRESS, t)
    ]

    # Rule: the receiver passes on a similar amount (within 10%) within an hour,
    # a classic sign of money being layered through an account.
    sent_values = {a: g.VALUE.to_numpy() for a, g in df.groupby("FROM_ADDRESS")}

    def forwards_similar(account, ti, amount):
        times = sends.get(account)
        if times is None:
            return False
        lo, hi = np.searchsorted(times, ti, "right"), np.searchsorted(times, ti + HOUR, "right")
        later = sent_values[account][lo:hi]
        return bool(np.any(np.abs(later - amount) <= 0.1 * amount))

    f["receiver_forwards_similar_1h"] = [
        forwards_similar(a, ti, v) for a, ti, v in zip(df.TO_ADDRESS, t, value)
    ]

    # Rule: the sender received a similar amount within the hour before.
    got_values = {a: g.VALUE.to_numpy() for a, g in df.groupby("TO_ADDRESS")}

    def received_similar(account, ti, amount):
        times = gets.get(account)
        if times is None:
            return False
        lo, hi = np.searchsorted(times, ti - HOUR, "left"), np.searchsorted(times, ti, "left")
        earlier = got_values[account][lo:hi]
        return bool(np.any(np.abs(earlier - amount) <= 0.1 * amount))

    f["sender_received_similar_1h"] = [
        received_similar(a, ti, v) for a, ti, v in zip(df.FROM_ADDRESS, t, value)
    ]

    # Rule: how quickly does the receiver send money on? (minutes, capped at 60)
    def minutes_to_next_send(account, ti):
        times = sends.get(account)
        if times is None:
            return 60.0
        i = np.searchsorted(times, ti, "right")
        return min(60.0, (times[i] - ti) / 60) if i < len(times) else 60.0

    f["receiver_minutes_to_forward"] = [
        minutes_to_next_send(a, ti) for a, ti in zip(df.TO_ADDRESS, t)
    ]

    # Rule: pass-through accounts send on most of what they receive over the week.
    sent_total = df.groupby("FROM_ADDRESS").VALUE.sum()
    got_total = df.groupby("TO_ADDRESS").VALUE.sum()
    pass_through = (sent_total / got_total.reindex(sent_total.index)).clip(0, 3)
    f["sender_pass_through"] = df.FROM_ADDRESS.map(pass_through).fillna(0)
    f["receiver_pass_through"] = df.TO_ADDRESS.map(pass_through).fillna(0)

    # Preserve Person 1's four aggregated transfer-count features.
    for column in ["transfer_count", "unique_senders", "unique_receivers", "unique_tokens", "hour", "day_of_week", "is_weekend"]:
        f[column] = df[column].to_numpy()

    # ---- Add your own rules here, for example: ----
    # f["round_amount"] = (value % 100 < 1)
    # f["night_time"] = (t % 86400) < 6 * 3600

    return f.astype({c: float for c in f.columns if c != "TRANSACTION_HASH"})


train_behaviour = load_for_ml(OUTPUT_DIR / "train_behaviour_features.parquet", OUTPUT_DIR.resolve().parent / "data/processed/train_cleaned.parquet", train.TRANSACTION_HASH)
train_features = make_features(train).merge(train_behaviour, on="TRANSACTION_HASH", validate="one_to_one")
y = train.set_index("TRANSACTION_HASH").is_suspicious.loc[train_features.TRANSACTION_HASH].to_numpy()
X = train_features.drop(columns="TRANSACTION_HASH")
X.describe().T[["mean", "max"]]

## 3. Train a logistic regression and check it

We hold back 20% of the training data, train on the rest, and score the held-back part the same way the site scores you: **100 × average precision**. Higher is better; guessing at random scores about 3.

In [ ]:
X_fit, X_check, y_fit, y_check = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

model = make_pipeline(
    SimpleImputer(strategy="constant", fill_value=-1, add_indicator=True),
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000),
)
model.fit(X_fit, y_fit)

check_scores = model.predict_proba(X_check)[:, 1]
print(f"Score on held-back training data: {100 * average_precision_score(y_check, check_scores):.1f}")

### Which rules mattered?

A positive weight means the rule makes a payment look **more** suspicious; a negative weight, less. The size shows how much it counts (features are scaled, so weights are comparable).

In [ ]:
weights = pd.Series(model[-1].coef_[0], index=model[:-1].get_feature_names_out(X.columns)).sort_values()
weights.round(2)

## 4. Score the test data and save your predictions

Retrain on all the training data, score every test transaction, and save `predictions.csv`. Upload it at https://hackathon.ukcbt.org together with a ZIP of your code.

In [ ]:
model.fit(X, y)

test = load(TEST_DIR)
test_behaviour = load_for_ml(OUTPUT_DIR / "test_behaviour_features.parquet", OUTPUT_DIR.resolve().parent / "data/processed/test_cleaned.parquet", test.TRANSACTION_HASH)
test_features = make_features(test).merge(test_behaviour, on="TRANSACTION_HASH", validate="one_to_one")
predictions = pd.DataFrame({
    "TRANSACTION_HASH": test_features.TRANSACTION_HASH,
    "score": model.predict_proba(test_features.drop(columns="TRANSACTION_HASH"))[:, 1],
})
predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print(f"Saved {len(predictions):,} predictions to ../outputs/predictions.csv")
predictions.head()

## Next steps

- **Add rules** in step 2, such as amounts that are suspiciously round, money that comes back to where it started, or accounts that only ever pass money on. Rerun, and keep the ones that raise your held-back score.
- **Try other models**: swap `LogisticRegression` for `sklearn.ensemble.HistGradientBoostingClassifier`, which can learn combinations of rules by itself.
- **Look further back**: the hard cases play out over several payments and changing counterparties, beyond the one-hour windows above.

Remember to include this notebook (or your scripts), a README with the exact command, and your dependencies in the code ZIP you upload.